In [ ]:
# ==============================================================================
# PIPELINE DE ENTRENAMIENTO DE ALTA PRECISIÓN - ALPR SYSTEM (YOLOv8 MLOps)
# ==============================================================================

# 1. Instalación de dependencias clave
!pip install -q ultralytics roboflow google-colab

import os
import torch
from ultralytics import YOLO
from roboflow import Roboflow
from google.colab import files, drive

# 2. Verificación y asignación de Hardware GPU
device = 0 if torch.cuda.is_available() else 'cpu'
print(f"--- 🚀 Dispositivo de Entrenamiento Detectado: {torch.cuda.get_device_name(0) if device == 0 else 'CPU'} ---")

# 3. Descarga estructurada del Dataset desde Roboflow
print("--- 📦 Descargando Dataset desde Roboflow... ---")
rf = Roboflow(api_key="zXYMFTVdrbIq01s81rNO")
project = rf.workspace("haeun-kim-ri91b").project("license-plate-detection-wienp")
version = project.version(2)
dataset = version.download("yolov8")

# 4. Carga del Modelo Base Potente (YOLOv8 Medium)
print("--- 🧠 Cargando pesos de arquitectura YOLOv8m... ---")
model = YOLO('yolov8m.pt')

# 5. Ejecución del Entrenamiento con Hiperparámetros Optimizados
print("--- 🔥 Iniciando Fine-Tuning de Alta Precisión... ---")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,                  # Límite superior suficiente con Early Stopping
    patience=20,                 # Detiene el entrenamiento si val_loss se estanca
    imgsz=800,                   # Mayor definición espacial para placas pequeñas
    batch=16,                    # Tamaño de lote (Reducir a 8 si Colab se queda sin VRAM)
    device=device,
    workers=4,

    # Optimizador y tasas de aprendizaje
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,

    # Data Augmentation Severo (Inmunidad a ángulos y sombras)
    degrees=12.0,                # Rotación de cámara
    translate=0.1,               # Traslación lateral
    scale=0.5,                   # Escalado/Zoom
    shear=2.0,                   # Deformación angular
    perspective=0.0005,          # Ángulos de disparo en vivo
    hsv_h=0.015,                 # Tono de color
    hsv_s=0.7,                   # Saturación (sol / sombra)
    hsv_v=0.4,                   # Brillo (nocturno / diurno)
    mosaic=1.0,                  # Combinación de imágenes

    # Guardado de checkpoints
    project='ALPR_UNAB_MLOps',
    name='yolov8m_production',
    save=True
)

# 6. Identificación del mejor archivo de pesos (.pt)
best_model_path = f"{results.save_dir}/weights/best.pt"
print(f"--- ✅ Entrenamiento Finalizado. Pesos óptimos guardados en: {best_model_path} ---")

# 7. Respaldo Automático en Google Drive (Buenas Prácticas MLOps)
try:
    print("--- 💾 Respaldando 'best.pt' en Google Drive... ---")
    drive.mount('/content/drive', force_remount=True)
    drive_backup_dir = '/content/drive/MyDrive/ALPR_Model_Backup'
    os.makedirs(drive_backup_dir, exist_ok=True)
    !cp {best_model_path} {drive_backup_dir}/best.pt
    print(f"--- 🟢 Copia guardada en Drive: {drive_backup_dir}/best.pt ---")
except Exception as e:
    print(f"⚠️ No se pudo realizar el respaldo automático en Drive: {e}")

# 8. Descarga Directa al Computador Host
print("--- ⬇️ Iniciando descarga automática del archivo 'best.pt'... ---")
files.download(best_model_path)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 100.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
--- 🚀 Dispositivo de Entrenamiento Detectado: Tesla T4 ---
--- 📦 Descargando Dat


Extracting Dataset Version Zip to license-plate-detection-2 in yolov8:: 100%|██████████| 11144/11144 [00:01<00:00, 6221.19it/s]


--- 🧠 Cargando pesos de arquitectura YOLOv8m... ---
--- 🔥 Iniciando Fine-Tuning de Alta Precisión... ---
Ultralytics 8.4.153 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/license-plate-detection-2/data.yaml, degrees=12.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mod

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>